In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H15 — Escalating Chain: Gravity → Recovery BPR → Re-ignite → Repeat
# ══════════════════════════════════════════════════════════════════════
#
# CONCEPT: Three-stage pipeline that fully exhausts each phase:
#
#   1. GRAVITY: Continuous optimisation (particles flowing on energy landscape)
#   2. RECOVERY BPR: Enhanced chain-death BPR (stubbornness targeting,
#      tree-walk, cooling chains) — exhaustively mops up everything
#      gravity almost solved
#   3. GRAVITY RE-IGNITION: If recovery failed → re-run gravity itself
#      seeded from best position, with ESCALATING NOISE to force wider
#      basin escape each round
#
# KEY IMPROVEMENTS over H14:
#
#   H14 Problem: Re-ignition used 80% seed / σ=0.15 every round
#   → Particles converged back to the same basin → R3/R4 added nothing
#
#   H15 Solution: ESCALATING aggressiveness per round:
#     R1: Full gravity (random)     → Recovery BPR (200K)
#     R2: 50% seed, σ=0.30, 750 ptcl → Recovery BPR (200K)
#     R3: 40% seed, σ=0.40, 750 ptcl → Recovery BPR (250K)
#     R4: 30% seed, σ=0.50, 750 ptcl → Recovery BPR (250K)
#     R5: 25% seed, σ=0.60, 750 ptcl → Recovery BPR (300K)
#     R6: 20% seed, σ=0.70, 750 ptcl → Recovery BPR (300K)
#     R7: 15% seed, σ=0.80, 1000 ptcl→ Recovery BPR (300K)
#     R8: 10% seed, σ=0.90, 1000 ptcl→ Recovery BPR (300K)
#
#   Each round: wider noise, lower seed fraction → forces exploration
#   of progressively more distant basins. By R8 it's nearly random
#   but with 10% anchored near the global-best-ever assignment.
#
# RECOVERY BPR fully exploits each gravity round before giving up:
#   • Chain death + stubbornness targeting (flip top 8% most stubborn vars)
#   • Tree-walk clause selection (branch_patience=80)
#   • Cooling chains (T × 0.95^k)
#   • Clause weighting (bump unsat + decay)
#   • Zero-break priority
#   • Top-5 particles × 3 weight modes (ensemble)
#
# GLOBAL BEST: Tracks the best-ever assignment across ALL rounds.
#   Re-ignition always seeds from global best, not just last round.
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time, math, os
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

N_WORKERS = min(os.cpu_count() or 2, 8)
GRAVITY_BATCH = 10

# ── Escalation schedule ───────────────────────────────────────────────
# (seed_frac, seed_noise, particles, bpr_flips) for rounds 2..8
ESCALATION = [
    # R2:  moderate exploration
    {'seed_frac': 0.50, 'noise': 0.30, 'particles': 750,  'flips': 200000},
    # R3:  wider
    {'seed_frac': 0.40, 'noise': 0.40, 'particles': 750,  'flips': 250000},
    # R4:  wider still
    {'seed_frac': 0.30, 'noise': 0.50, 'particles': 750,  'flips': 250000},
    # R5:  aggressive
    {'seed_frac': 0.25, 'noise': 0.60, 'particles': 750,  'flips': 300000},
    # R6:  very aggressive
    {'seed_frac': 0.20, 'noise': 0.70, 'particles': 750,  'flips': 300000},
    # R7:  near-random with anchor
    {'seed_frac': 0.15, 'noise': 0.80, 'particles': 1000, 'flips': 300000},
    # R8:  almost fully random restart with 10% anchor
    {'seed_frac': 0.10, 'noise': 0.90, 'particles': 1000, 'flips': 300000},
]
MAX_ROUNDS = len(ESCALATION) + 1  # +1 for Round 1


# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  TREE-WALK BPR with Chain Death + Stubbornness (RECOVERY PHASE)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_chain(clauses_v, clauses_s, assignment, weight,
              max_flips=200000, T_init=0.5, T_min=0.01,
              p_random=0.1, beta=0.3,
              chain_patience=5000, branch_patience=80,
              cool_rate=0.95, stub_frac=0.08,
              weight_bump=2.0, weight_decay=0.9):
    """
    Tree-Walk BPR with adaptive chain death + stubbornness re-ignition.
    RECOVERY PHASE: exhaustively exploits the current gravity weights.
    Returns: (best_assignment, flips_used, n_chains, best_n_unsat)
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    n_stub = max(2, int(stub_frac * n))

    # ── Build adjacency ──
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    clause_w = np.ones(m, dtype=np.float64)

    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    best_n_unsat = n_unsat
    best_assign = assignment.copy()

    chain_id = 0
    chain_stale = 0
    chain_best_unsat = n_unsat

    in_branch = False
    branch_stale = 0
    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0
    stubbornness = np.zeros(n, dtype=np.float64)

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip, chain_id + 1, 0

        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]
            chain_stale = 0
            chain_best_unsat = n_unsat
        elif n_unsat < chain_best_unsat:
            chain_best_unsat = n_unsat
            chain_stale = 0
        else:
            chain_stale += 1

        # ── CHAIN DEATH + RE-IGNITION ──
        if chain_stale >= chain_patience and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += weight_bump
            for c in range(m):
                clause_w[c] *= weight_decay

            for vi in range(n):
                stubbornness[vi] = 0.0
            for ui in range(n_unsat):
                cc = unsat_list[ui]
                w_c = clause_w[cc]
                for j in range(3):
                    stubbornness[clauses_v[cc, j]] += w_c

            stub_to_flip = np.zeros(n_stub, dtype=np.int32)
            stub_used = np.zeros(n, dtype=np.int8)
            for k in range(n_stub):
                best_sv = -1.0
                best_vi = 0
                for vi in range(n):
                    if stub_used[vi] == 0 and stubbornness[vi] > best_sv:
                        best_sv = stubbornness[vi]
                        best_vi = vi
                stub_to_flip[k] = best_vi
                stub_used[best_vi] = 1

            for i in range(n):
                assignment[i] = best_assign[i]
            for k in range(n_stub):
                assignment[stub_to_flip[k]] = 1 - assignment[stub_to_flip[k]]

            n_random = max(1, n // 100)
            for _ in range(n_random):
                vi = np.random.randint(n)
                if np.random.random() < 0.3:
                    assignment[vi] = 1 - assignment[vi]

            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1

            if n_unsat == 0:
                return assignment, flip, chain_id + 1, 0

            chain_id += 1
            chain_stale = 0
            chain_best_unsat = n_unsat
            in_branch = False
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0
            continue

        # ── TREE-WALK CLAUSE SELECTION ──
        ci = -1
        if in_branch and branch_stale < branch_patience:
            best_neighbor_w = -1.0
            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc
            if ci < 0:
                in_branch = False

        if not in_branch or ci < 0:
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        T_max_chain = T_init * (cool_rate ** min(chain_id, 30))
        local_prog = min(1.0, chain_stale / chain_patience)
        T = T_min + (T_max_chain - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        # ── VARIABLE SELECTION ──
        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)
            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0; w_brk = 0.0; w_make = 0.0
                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]
                int_brks[j] = i_brk
                delta = w_brk - w_make
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1
            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        # ── Execute flip ──
        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]
        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        if n_unsat < old_n_unsat:
            branch_stale = 0
        else:
            branch_stale += 1

        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips, chain_id + 1, best_n_unsat


# ══════════════════════════════════════════════════════════════════════
#  NUMBA SAT CHECK + UNSAT COUNTER
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def check_sat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            return False
    return True

@njit(cache=True)
def count_unsat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    cnt = 0
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            cnt += 1
    return cnt


# ══════════════════════════════════════════════════════════════════════
#  BATCHED ENERGY
# ══════════════════════════════════════════════════════════════════════

def _energy_batched(s, mu_val, vars_t, signs_t):
    B, P, n = s.shape
    m = vars_t.shape[1]
    prod_val = torch.ones(B, P, m, device=s.device, dtype=s.dtype)
    for j in range(3):
        idx_j = vars_t[:, :, j].unsqueeze(1).expand(B, P, m)
        gathered_j = torch.gather(s, 2, idx_j)
        lit_j = gathered_j * signs_t[:, None, :, j]
        prod_val = prod_val * (1.0 - lit_j)
    e_sat = (prod_val / 8.0).sum(dim=-1)
    if mu_val > 0:
        return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(dim=-1)
    return e_sat


# ══════════════════════════════════════════════════════════════════════
#  GRAVITY CORE — shared by initial + re-ignition
# ══════════════════════════════════════════════════════════════════════

def _gravity_core(s, vars_t, signs_t, steps, lr=0.05, momentum_beta=0.9,
                  mu_scale=0.1, G_max=0.10, top_k_frac=0.1,
                  gravity_start=0.2, elite_repulsion=0.5,
                  gravity_interval=20):
    """Run gravity flow on pre-initialised particles s (B, P, n)."""
    B, particles, n = s.shape
    gi = gravity_interval
    use_amp = (device.type == 'cuda')

    vel = torch.zeros_like(s)
    grav_step = int(gravity_start * steps)
    delay_step = int(0.7 * steps)
    top_k = max(1, int(top_k_frac * particles))
    theta = None

    best_e = torch.full((B, particles), float('inf'), device=device)
    plateau_count = torch.zeros(B, particles, device=device)
    decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(
        steps, device=device, dtype=torch.float32))
    cached_targets = [None] * B

    for step in range(steps):
        if step < delay_step:
            mu_val = 0.0
        else:
            t_l = (step - delay_step) / (steps - delay_step)
            mu_val = mu_scale * 0.5 * (1.0 - math.cos(math.pi * t_l))

        s = s.detach().requires_grad_(True)
        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                e = _energy_batched(s, mu_val, vars_t, signs_t)
                e_f32 = e.float()
        else:
            e_f32 = _energy_batched(s, mu_val, vars_t, signs_t)

        e_vals = e_f32.detach()
        e_f32.sum().backward()
        g = s.grad.detach().clone()

        with torch.no_grad():
            decay = decay_arr[step]
            improved = e_vals < best_e
            best_e = torch.where(improved, e_vals, best_e)
            plateau_count = torch.where(improved,
                torch.zeros_like(plateau_count), plateau_count + 1)
            pm = (plateau_count >= 50).unsqueeze(2)

            gnorm = g.norm(dim=2, keepdim=True).clamp_(min=1e-10)
            dte = (lr * decay) / (1.0 + 0.05 * gnorm)
            dte = dte * torch.where(pm, torch.tensor(2.0, device=device),
                                         torch.tensor(1.0, device=device))

            if theta is None:
                theta = float(e_vals.median()) + 1e-8
            damp = 1.0 / (1.0 + e_vals.unsqueeze(2) / theta)
            gam = (e_vals.clamp(min=0) / (e_vals + 1.0)).unsqueeze(2)

            vel = momentum_beta * vel - dte * damp * (1.0 + gam) * g

            ns_base = 0.03 * decay
            noise = torch.randn_like(s) * torch.where(pm, 4.0 * ns_base, ns_base)
            s = (s + vel + noise).clamp_(-1, 1)

            if step >= grav_step and (step - grav_step) % gi == 0:
                progress = (step - grav_step) / (steps - grav_step)
                g_mag = G_max * progress * progress * gi
                for b in range(B):
                    _, top_idx = e_vals[b].topk(top_k, largest=False)
                    elite = s[b, top_idx]
                    d = torch.cdist(s[b].unsqueeze(0), elite.unsqueeze(0))[0]
                    cached_targets[b] = elite[d.argmin(dim=1)]
                    if top_k > 1:
                        ed = torch.cdist(elite.unsqueeze(0), elite.unsqueeze(0))[0]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[b, top_idx] += (elite_repulsion * g_mag) * (push / pn)
                    s[b].add_(g_mag * (cached_targets[b] - s[b]))
            elif step >= grav_step:
                for b in range(B):
                    if cached_targets[b] is not None:
                        progress = (step - grav_step) / (steps - grav_step)
                        g_mag = G_max * progress * progress
                        s[b].add_(g_mag * (cached_targets[b] - s[b]))

            s.clamp_(-1, 1)
            if (step + 1) % 200 == 0:
                theta = float(e_vals.median()) + 1e-8

    return s.detach()


# ══════════════════════════════════════════════════════════════════════
#  INITIAL GRAVITY (full, from random)
# ══════════════════════════════════════════════════════════════════════

def batched_gravity_flow(all_instances, n, m, steps=4000, particles=1000,
                         batch_size=None):
    if batch_size is None:
        batch_size = GRAVITY_BATCH

    N_INST = len(all_instances)
    all_results = [None] * N_INST

    for batch_start in range(0, N_INST, batch_size):
        batch_end = min(batch_start + batch_size, N_INST)
        B = batch_end - batch_start
        batch_instances = all_instances[batch_start:batch_end]

        vars_list = []
        signs_list = []
        for clauses in batch_instances:
            vs = [v for v, s in clauses]
            ss = [s for v, s in clauses]
            vars_list.append(vs)
            signs_list.append(ss)

        vars_t = torch.tensor(vars_list, dtype=torch.long, device=device)
        signs_t = torch.tensor(signs_list, dtype=torch.float32, device=device)

        s = (torch.randn(B, particles, n, device=device) * 0.3).clamp_(-0.9, 0.9)
        s_final = _gravity_core(s, vars_t, signs_t, steps=steps)

        for b in range(B):
            all_results[batch_start + b] = s_final[b]

        del s_final, vars_t, signs_t
        torch.cuda.empty_cache() if device.type == 'cuda' else None

    return all_results


# ══════════════════════════════════════════════════════════════════════
#  SEEDED GRAVITY RE-IGNITION — escalating noise
# ══════════════════════════════════════════════════════════════════════

def seeded_gravity_flow(seed_assignments, all_instances, n, m,
                        steps=2000, particles=750,
                        seed_frac=0.50, seed_noise=0.30,
                        batch_size=None):
    """
    Re-ignition gravity with escalating parameters.
    seed_frac and seed_noise change each round (via ESCALATION schedule).
    """
    if batch_size is None:
        batch_size = GRAVITY_BATCH

    N_INST = len(all_instances)
    all_results = [None] * N_INST
    n_seeded = int(seed_frac * particles)
    n_random = particles - n_seeded

    for batch_start in range(0, N_INST, batch_size):
        batch_end = min(batch_start + batch_size, N_INST)
        B = batch_end - batch_start
        batch_instances = all_instances[batch_start:batch_end]
        batch_seeds = seed_assignments[batch_start:batch_end]

        vars_list = []
        signs_list = []
        for clauses in batch_instances:
            vs = [v for v, s in clauses]
            ss = [s for v, s in clauses]
            vars_list.append(vs)
            signs_list.append(ss)

        vars_t = torch.tensor(vars_list, dtype=torch.long, device=device)
        signs_t = torch.tensor(signs_list, dtype=torch.float32, device=device)

        # ── SEEDED INITIALIZATION ──
        s_seed = torch.zeros(B, n, device=device)
        for b in range(B):
            for i in range(n):
                s_seed[b, i] = 0.7 if batch_seeds[b][i] == 1 else -0.7

        # Seeded particles: near seed with escalating noise
        s_hot = s_seed.unsqueeze(1).expand(B, n_seeded, n).clone()
        s_hot += torch.randn(B, n_seeded, n, device=device) * seed_noise
        s_hot.clamp_(-0.9, 0.9)

        # Random particles: fully random for diversity
        s_cold = (torch.randn(B, n_random, n, device=device) * 0.3).clamp_(-0.9, 0.9)

        # Combine
        s = torch.cat([s_hot, s_cold], dim=1)  # (B, P, n)

        s_final = _gravity_core(s, vars_t, signs_t, steps=steps)

        for b in range(B):
            all_results[batch_start + b] = s_final[b]

        del s, s_final, vars_t, signs_t
        torch.cuda.empty_cache() if device.type == 'cuda' else None

    return all_results


# ══════════════════════════════════════════════════════════════════════
#  PER-INSTANCE HELPER
# ══════════════════════════════════════════════════════════════════════

class InstanceHelper:
    def __init__(self, n, clauses):
        self.n = n
        self.m = len(clauses)
        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t = torch.tensor(vs_list, dtype=torch.long, device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)

    def find_best_particles(self, s, k=5):
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            topk_sat, topk_idx = n_sat.topk(k, largest=True)
            viols = self.m - topk_sat
            return topk_idx.cpu().tolist(), viols.cpu().tolist()

    def get_confidence(self, s, idx):
        with torch.no_grad():
            return (1.0 - s[idx].abs()).cpu().numpy().astype(np.float64)

    def get_mfls(self, s, idx):
        sb = s[idx].clone().detach().requires_grad_(True)
        lit = sb[self.vars_t] * self.signs_t
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum()
        grad = torch.autograd.grad(e_sat, sb)[0]
        g = grad.abs().cpu().numpy().astype(np.float64)
        mx = g.max()
        if mx < 1e-10:
            return np.full(self.n, 0.5, dtype=np.float64)
        return g / mx

    def get_quadsurf(self, s, idx):
        with torch.no_grad():
            sb = s[idx]
            lit = sb[self.vars_t] * self.signs_t
            clause_e = (1.0 - lit).prod(dim=1) / 8.0
            qs = torch.zeros(self.n, device=device)
            for j in range(3):
                qs.scatter_add_(0, self.vars_t[:, j], clause_e)
            qs = qs.cpu().numpy().astype(np.float64)
            mx = qs.max()
            if mx < 1e-10:
                return np.full(self.n, 0.5, dtype=np.float64)
            return qs / mx


# ══════════════════════════════════════════════════════════════════════
#  BPR WORKER
# ══════════════════════════════════════════════════════════════════════

def bpr_worker(clauses_v, clauses_s, x_np, weight,
               max_flips, T_init, T_min, p_random, beta,
               chain_patience, branch_patience,
               cool_rate, stub_frac, weight_bump, weight_decay):
    sol, flips, n_chains, remaining = bpr_chain(
        clauses_v, clauses_s, x_np, weight,
        max_flips=max_flips, T_init=T_init, T_min=T_min,
        p_random=p_random, beta=beta,
        chain_patience=chain_patience, branch_patience=branch_patience,
        cool_rate=cool_rate, stub_frac=stub_frac,
        weight_bump=weight_bump, weight_decay=weight_decay
    )
    is_sat = check_sat(clauses_v, clauses_s, sol)
    return is_sat, flips, n_chains, sol


# ══════════════════════════════════════════════════════════════════════
#  RECOVERY PHASE: Run BPR on all top-K particles × 3 modes
#  Returns: (any_solved, best_partial_assignment, best_unsat_count)
# ══════════════════════════════════════════════════════════════════════

def run_recovery_bpr(eng, sf, max_flips, top_k=5, beta=0.3,
                     chain_patience=5000, branch_patience=80,
                     cool_rate=0.95, stub_frac=0.08,
                     weight_bump=2.0, weight_decay=0.9):
    """
    Full recovery phase: try top-K particles × 3 weight modes in parallel.
    Returns (solved, best_assignment_np, best_unsat_count, per_mode_solved)
    """
    modes = ['confidence', 'mfls', 'quadsurf']
    top_indices, top_viols = eng.find_best_particles(sf, k=top_k)

    any_solved = False
    best_sol = None
    best_unsat = eng.m  # track global best partial
    mode_solved = {m: False for m in modes}

    # Check gravity-only solves first
    for pi, viols in zip(top_indices, top_viols):
        if viols == 0:
            x_np = (sf[pi] > 0).cpu().numpy().astype(np.int32)
            return True, x_np, 0, {m: True for m in modes}, True  # grav_only=True

    for rank, (pi, viols) in enumerate(zip(top_indices, top_viols)):
        if any_solved and rank > 0:
            break

        x_np = (sf[pi] > 0).cpu().numpy().astype(np.int32)
        w_conf = eng.get_confidence(sf, pi)
        w_mfls = eng.get_mfls(sf, pi)
        w_qs = eng.get_quadsurf(sf, pi)
        weights = {'confidence': w_conf, 'mfls': w_mfls, 'quadsurf': w_qs}

        futures = {}
        with ThreadPoolExecutor(max_workers=min(N_WORKERS, 3)) as pool:
            for md in modes:
                f = pool.submit(
                    bpr_worker, eng.clauses_v, eng.clauses_s,
                    x_np.copy(), weights[md],
                    max_flips, 0.5, 0.01, 0.1, beta,
                    chain_patience, branch_patience,
                    cool_rate, stub_frac, weight_bump, weight_decay
                )
                futures[f] = md

            for f in as_completed(futures):
                md = futures[f]
                is_sat, flips, n_chains, sol = f.result()
                if is_sat:
                    if rank == 0:
                        mode_solved[md] = True
                    any_solved = True

                # Track best partial (using numba count)
                sol_unsat = count_unsat(eng.clauses_v, eng.clauses_s, sol)
                if sol_unsat < best_unsat:
                    best_unsat = sol_unsat
                    best_sol = sol.copy()

        if any_solved:
            break

    if best_sol is None:
        # Fallback: use top particle discretised
        best_sol = (sf[top_indices[0]] > 0).cpu().numpy().astype(np.int32)
        best_unsat = count_unsat(eng.clauses_v, eng.clauses_s, best_sol)

    return any_solved, best_sol, best_unsat, mode_solved, False  # grav_only=False


# ══════════════════════════════════════════════════════════════════════
#  COMPILE + WARMUP
# ══════════════════════════════════════════════════════════════════════

_wv = np.array([[0, 1, 2]], dtype=np.int32)
_ws = np.array([[1, -1, 1]], dtype=np.int32)
_wa = np.array([1, 0, 1], dtype=np.int32)
_ww = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_chain(_wv, _ws, _wa, _ww, max_flips=100,
              chain_patience=20, branch_patience=5)
_ = check_sat(_wv, _ws, _wa)
_ = count_unsat(_wv, _ws, _wa)

print(f'CPU cores: {os.cpu_count()}, BPR workers: {N_WORKERS}')
print()
print('✓ H15 — Escalating Chain: Gravity → Recovery BPR → Re-ignite → Repeat')
print(f'  Device:  {device}')
if device.type == 'cuda':
    print(f'  GPU:     {torch.cuda.get_device_name()}')
    print(f'  VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print()
print('  Pipeline per instance:')
print('    ┌─ R1: GRAVITY (4000 steps × 1000 particles, random init)')
print('    │   → RECOVERY BPR (200K flips, chains+stubbornness, 3 modes)')
print('    │   → Solved? DONE. Failed? → global_best = best partial')
print('    │')
for i, esc in enumerate(ESCALATION):
    rnd = i + 2
    last = (i == len(ESCALATION) - 1)
    prefix = '└' if last else '├'
    print(f'    {prefix}─ R{rnd}: RE-IGNITE GRAVITY '
          f'({int(esc["seed_frac"]*100)}% seed, σ={esc["noise"]:.1f}, '
          f'{esc["particles"]} ptcl)')
    print(f'    {"  " if last else "│"}   → RECOVERY BPR '
          f'({esc["flips"]//1000}K flips) → Solved? DONE.')
print()
print('  Escalation: seed fraction ↓ (50→10%), noise ↑ (0.3→0.9)')
print('  Each round explores wider basins. Recovery BPR fully exhausts each.')
print(f'  Max {MAX_ROUNDS} rounds per instance.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H15 Experiment: Escalating Gravity → Recovery BPR → Re-ignite → Repeat
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50

# Round 1 parameters
PARTICLES_R1 = 1000
STEPS_R1     = 4000
FLIPS_R1     = 200000
STEPS_RN     = 2000  # re-ignition gravity steps

BETA     = 0.3
TOP_K_PARTICLES = 5
BRANCH_PATIENCE = 80
CHAIN_PATIENCE  = 5000
COOL_RATE       = 0.95
STUB_FRAC       = 0.08
WEIGHT_BUMP     = 2.0
WEIGHT_DECAY    = 0.9

MODES = ['confidence', 'mfls', 'quadsurf']

# All baselines
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h11b_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 68.0,  (4.0, 750): 76.0,  (4.0, 1000): 52.0,
    (4.2, 500):  8.0,  (4.2, 750):  2.0,  (4.2, 1000):  2.0,
}
h13_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 94.0,  (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 30.0,  (4.2, 750): 32.0,  (4.2, 1000): 16.0,
}
h14_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 42.0,  (4.2, 750): 28.0,  (4.2, 1000): 20.0,
}

results = {m: {} for m in MODES}
ensemble_results = {}
s1_results = {}
round_stats = {}

print('=' * 140)
print('H15 — Escalating Chain: Gravity → Recovery BPR → Re-ignite → Repeat')
print(f'  Round 1: {STEPS_R1} steps × {PARTICLES_R1} particles (random) → Recovery BPR ({FLIPS_R1//1000}K)')
for i, esc in enumerate(ESCALATION):
    rnd = i + 2
    print(f'  Round {rnd}: {STEPS_RN} steps × {esc["particles"]} particles '
          f'({int(esc["seed_frac"]*100)}% seed, σ={esc["noise"]:.1f}) '
          f'→ Recovery BPR ({esc["flips"]//1000}K)')
print(f'  Max {MAX_ROUNDS} rounds | Top-{TOP_K_PARTICLES} particles × {len(MODES)} modes')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'S1':>3} | {'Conf%':>6} | {'MFLS%':>6} | "
      f"{'QS%':>6} | {'Ens%':>6} | {'H14E':>5} | {'H13E':>5} |"
      f" {'ΔvsH14':>7} | {'AvgRnd':>6} | Time")
print('  ' + '-' * 128)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        # ═══════════════════════════════════════════════════════════
        # PHASE 1: Generate all instances
        # ═══════════════════════════════════════════════════════════
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        helpers = [InstanceHelper(n_var, all_instances[i]) for i in range(N_INST)]

        # ═══════════════════════════════════════════════════════════
        # TRACKING
        # ═══════════════════════════════════════════════════════════
        inst_solved = [False] * N_INST
        inst_round_solved = [0] * N_INST
        inst_global_best = [None] * N_INST  # GLOBAL best assignment across all rounds
        inst_global_best_unsat = [m_cls] * N_INST  # how many unsat for global best
        s1_count = 0
        mode_ok = {m: 0 for m in MODES}
        ens_ok = 0

        # ═══════════════════════════════════════════════════════════
        # ROUND 1: FULL GRAVITY → RECOVERY BPR
        # ═══════════════════════════════════════════════════════════
        t_r1_start = time.time()
        print(f'    α={alpha}, n={n_var}: R1 gravity '
              f'({STEPS_R1} steps × {PARTICLES_R1} ptcl)...')

        all_particles = batched_gravity_flow(
            all_instances, n_var, m_cls,
            steps=STEPS_R1, particles=PARTICLES_R1,
            batch_size=GRAVITY_BATCH
        )

        print(f'    → R1 gravity done ({time.time()-t_r1_start:.0f}s). Running recovery BPR...')

        for inst in range(N_INST):
            solved, best_sol, best_unsat, msolved, grav_only = run_recovery_bpr(
                helpers[inst], all_particles[inst],
                max_flips=FLIPS_R1, top_k=TOP_K_PARTICLES,
                beta=BETA, chain_patience=CHAIN_PATIENCE,
                branch_patience=BRANCH_PATIENCE,
                cool_rate=COOL_RATE, stub_frac=STUB_FRAC,
                weight_bump=WEIGHT_BUMP, weight_decay=WEIGHT_DECAY
            )

            if grav_only:
                s1_count += 1

            if solved:
                inst_solved[inst] = True
                inst_round_solved[inst] = 1
                for md in MODES:
                    if msolved[md]:
                        mode_ok[md] += 1
                    else:
                        mode_ok[md] += 1  # ensemble solved it
                ens_ok += 1
            else:
                # Update global best
                inst_global_best[inst] = best_sol
                inst_global_best_unsat[inst] = best_unsat

        n_solved_r1 = sum(inst_solved)
        t_r1 = time.time() - t0
        print(f'    → R1: {n_solved_r1}/{N_INST} solved ({t_r1:.0f}s)')

        # ═══════════════════════════════════════════════════════════
        # ROUNDS 2-8: ESCALATING RE-IGNITION → RECOVERY BPR
        # ═══════════════════════════════════════════════════════════
        for rnd_idx, esc in enumerate(ESCALATION):
            rnd = rnd_idx + 2

            unsolved_indices = [i for i in range(N_INST) if not inst_solved[i]]
            if len(unsolved_indices) == 0:
                break

            t_rn_start = time.time()
            sf = esc['seed_frac']
            sn = esc['noise']
            ptcl = esc['particles']
            flips = esc['flips']

            print(f'    → R{rnd}: Re-igniting gravity for '
                  f'{len(unsolved_indices)} unsolved '
                  f'({int(sf*100)}% seed, σ={sn:.1f}, {ptcl} ptcl)...')

            # Collect GLOBAL BEST seeds and instances
            rn_instances = [all_instances[i] for i in unsolved_indices]
            rn_seeds = [inst_global_best[i] for i in unsolved_indices]

            # ── SEEDED GRAVITY with escalating parameters ──
            rn_particles = seeded_gravity_flow(
                rn_seeds, rn_instances, n_var, m_cls,
                steps=STEPS_RN, particles=ptcl,
                seed_frac=sf, seed_noise=sn,
                batch_size=GRAVITY_BATCH
            )

            t_rn_grav = time.time() - t_rn_start

            # ── RECOVERY BPR ──
            rn_solved = 0
            for ui, orig_idx in enumerate(unsolved_indices):
                solved, best_sol, best_unsat, msolved, grav_only = run_recovery_bpr(
                    helpers[orig_idx], rn_particles[ui],
                    max_flips=flips, top_k=TOP_K_PARTICLES,
                    beta=BETA, chain_patience=CHAIN_PATIENCE,
                    branch_patience=BRANCH_PATIENCE,
                    cool_rate=COOL_RATE, stub_frac=STUB_FRAC,
                    weight_bump=WEIGHT_BUMP, weight_decay=WEIGHT_DECAY
                )

                if grav_only:
                    s1_count += 1

                if solved:
                    inst_solved[orig_idx] = True
                    inst_round_solved[orig_idx] = rnd
                    for md in MODES:
                        mode_ok[md] += 1
                    ens_ok += 1
                    rn_solved += 1
                else:
                    # Update GLOBAL best if this round found better partial
                    if best_unsat < inst_global_best_unsat[orig_idx]:
                        inst_global_best[orig_idx] = best_sol
                        inst_global_best_unsat[orig_idx] = best_unsat

            t_rn = time.time() - t_rn_start
            n_total_solved = sum(inst_solved)
            n_remaining = len(unsolved_indices) - rn_solved
            print(f'    → R{rnd}: +{rn_solved} solved '
                  f'({n_total_solved}/{N_INST} total, '
                  f'{n_remaining} remaining, {t_rn:.0f}s)')

            # Early exit if all solved
            if n_total_solved == N_INST:
                break

        # ═══════════════════════════════════════════════════════════
        # FINAL RESULTS for this (α, n)
        # ═══════════════════════════════════════════════════════════
        elapsed = time.time() - t0

        for md in MODES:
            pct = mode_ok[md] / N_INST * 100
            results[md][(alpha, n_var)] = {'pct': pct}

        ens_pct = ens_ok / N_INST * 100
        ensemble_results[(alpha, n_var)] = ens_pct
        s1_results[(alpha, n_var)] = s1_count

        # Round stats
        rs = {'avg': 0, 'unsolved': 0}
        for rr in range(1, MAX_ROUNDS + 1):
            rs[f'r{rr}'] = sum(1 for r in inst_round_solved if r == rr)
        rs['unsolved'] = sum(1 for s in inst_solved if not s)
        rounds_used = [inst_round_solved[i] for i in range(N_INST) if inst_solved[i]]
        rs['avg'] = np.mean(rounds_used) if rounds_used else 0
        round_stats[(alpha, n_var)] = rs

        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']

        h14e = h14_ens[(alpha, n_var)]
        h13e = h13_ens[(alpha, n_var)]
        delta_h14 = ens_pct - h14e

        tag = '★' if ens_pct >= 95 else ('▲' if delta_h14 > 2 else
              ('≈' if abs(delta_h14) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {s1_count:3d}  | '
              f'{c_:5.1f}% | {m_:5.1f}% | {q_:5.1f}% | {ens_pct:5.1f}% | '
              f'{h14e:4.0f}% | {h13e:4.0f}% | {delta_h14:+6.1f}% | '
              f'{rs["avg"]:5.1f} | {elapsed:.0f}s {tag}')
    print('  ' + '-' * 128)


# ══════════════════════════════════════════════════════════════════════
#  Full Summary
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('FULL COMPARISON — H9b → H10b → H11b → H13 → H14 → H15')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | {'H11bE':>6} | "
      f"{'H13E':>5} | {'H14E':>5} | {'H15C':>5} | {'H15M':>5} | {'H15Q':>5} | "
      f"{'H15 Ens':>8} | {'ΔvsH14':>7}")
print('  ' + '-' * 120)
for alpha in ALPHAS:
    for n_var in NS:
        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']
        e_ = ensemble_results[(alpha, n_var)]
        h9 = h9b[(alpha, n_var)]
        h10 = h10b[(alpha, n_var)]
        h11e = h11b_ens[(alpha, n_var)]
        h13e = h13_ens[(alpha, n_var)]
        h14e = h14_ens[(alpha, n_var)]
        delta = e_ - h14e
        print(f'  {alpha:5.1f} | {n_var:5d} | {h9:4.0f}% | {h10:4.0f}% | '
              f'{h11e:5.1f}% | {h13e:4.0f}% | {h14e:4.0f}% | '
              f'{c_:4.1f}% | {m_:4.1f}% | {q_:4.1f}% | '
              f'{e_:7.1f}% | {delta:+6.1f}%')
    print('  ' + '-' * 120)


# ══════════════════════════════════════════════════════════════════════
#  ROUND-BY-ROUND ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('ESCALATING RE-IGNITION ANALYSIS — which rounds solved instances?')
print('=' * 140)

# Header
hdr = f"  {'α':>5} | {'n':>5}"
for rr in range(1, MAX_ROUNDS + 1):
    hdr += f" | {'R'+str(rr):>4}"
hdr += f" | {'Fail':>4} | {'AvgRnd':>6} | {'Interpretation':>40}"
print(hdr)
print('  ' + '-' * 130)

for alpha in ALPHAS:
    for n_var in NS:
        rs = round_stats[(alpha, n_var)]
        row = f'  {alpha:5.1f} | {n_var:5d}'
        for rr in range(1, MAX_ROUNDS + 1):
            row += f' | {rs.get(f"r{rr}", 0):4d}'
        row += f' | {rs["unsolved"]:4d} | {rs["avg"]:5.1f}'

        # Interpretation
        total_solved = N_INST - rs['unsolved']
        r1_count = rs.get('r1', 0)
        later_count = total_solved - r1_count
        if rs['unsolved'] == 0:
            if r1_count == N_INST:
                interp = 'All R1 — easy'
            else:
                interp = f'R1={r1_count}, re-ignition rescued +{later_count}!'
        else:
            if later_count > 0:
                interp = f'R1={r1_count}, +{later_count} rescued, {rs["unsolved"]} too hard'
            else:
                interp = f'R1={r1_count} only, {rs["unsolved"]} too hard'
        row += f' | {interp}'
        print(row)
    print('  ' + '-' * 130)


# ══════════════════════════════════════════════════════════════════════
#  REMAINING UNSAT ANALYSIS (for failed instances)
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('UNSOLVED INSTANCE ANALYSIS — how close did we get?')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        unsolved = [i for i in range(N_INST) if not inst_solved[i]]
        if not unsolved:
            continue
        unsats = [inst_global_best_unsat[i] for i in unsolved]
        print(f'  α={alpha}, n={n_var}: {len(unsolved)} unsolved | '
              f'unsat: min={min(unsats)}, median={np.median(unsats):.0f}, '
              f'max={max(unsats)}, mean={np.mean(unsats):.1f}')


# ══════════════════════════════════════════════════════════════════════
#  Charts
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...\n')

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
w = 0.11
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#8e44ad', '#1abc9c', '#e67e22']
labels_c = ['H9b', 'H10b', 'H11b ens', 'H13 ens', 'H14 ens',
            'H15 best single', 'H15 ensemble']

for i, n_var in enumerate(NS):
    ax = axes[i]
    x = np.arange(len(ALPHAS))

    bars_data = [
        [h9b[(a, n_var)] for a in ALPHAS],
        [h10b[(a, n_var)] for a in ALPHAS],
        [h11b_ens[(a, n_var)] for a in ALPHAS],
        [h13_ens[(a, n_var)] for a in ALPHAS],
        [h14_ens[(a, n_var)] for a in ALPHAS],
        [max(results['confidence'][(a, n_var)]['pct'],
             results['mfls'][(a, n_var)]['pct'],
             results['quadsurf'][(a, n_var)]['pct']) for a in ALPHAS],
        [ensemble_results[(a, n_var)] for a in ALPHAS],
    ]

    for k, (label, vals) in enumerate(zip(labels_c, bars_data)):
        offset = (k - 3) * w
        ax.bar(x + offset, vals, w, label=label, color=colors[k], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=5.5, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H15: Escalating Chain — Gravity → Recovery BPR → Re-ignite (wider each round)\n'
             'Seed fraction ↓ (50→10%) | Noise ↑ (0.3→0.9) | '
             f'Up to {MAX_ROUNDS} rounds per instance',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('h15_escalate_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h15_escalate_results.png')

# ── Gravity-only solves ──
print('\nGravity-only solves (all rounds combined):')
for alpha in ALPHAS:
    for n_var in NS:
        s1 = s1_results[(alpha, n_var)]
        if s1 > 0:
            print(f'  α={alpha}, n={n_var}: {s1}/50 ★')

# ── Total flips budget used ──
print('\nEffective budget per instance (max possible):')
total_budget = FLIPS_R1
for esc in ESCALATION:
    total_budget += esc['flips']
print(f'  R1: {FLIPS_R1//1000}K + R2-R8: '
      f'{sum(e["flips"] for e in ESCALATION)//1000}K = '
      f'{total_budget//1000}K total max')